# Notebook 25 — Wu 2003 SBI Training: S-A Structure

Train a Neural Posterior Estimator (SNPE-C / NPE) on the **S-A** observation structure
(includes distillate-composition analyser — x_D in control loop).
Then compare posteriors with nb24's S-B posterior to quantify the information value of x_D.

**Contents:**
1. Generate S-A training data
2. Train SNPE on S-A summaries
3. Simulation-Based Calibration (SBC) for S-A
4. W12 headline comparison: S-A vs S-B posteriors
5. Information value of x_D: 90% CI width comparison across all 16 scenarios
6. Save S-A posterior

**Runtime notes:**
- Data generation with `N_TRAIN=1000`: ~5–15 min on CPU
- SNPE training: ~5–20 min
- CI width comparison (all 16 scenarios × 2 structures): ~10–20 min

In [ ]:
import sys; sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import pickle
import time
import pathlib
from scipy import stats as scipy_stats

from cstr_sbi.recycle.priors import box_uniform_5d, PRIOR_LOW_5D, PRIOR_HIGH_5D
from cstr_sbi.recycle.physics import (
    NOMINAL_CTRL_SA, NOMINAL_CTRL_SB, NOMINAL_INLET, NOMINAL_Y0_EXPLICIT, PARAM_NAMES,
    simulate_trajectory_explicit, extract_observations_explicit
)
from cstr_sbi.recycle.summaries import compute_summaries, N_SUMMARIES_SA, N_SUMMARIES_SB
from cstr_sbi.recycle.scenarios import CLOSED_LOOP_NAMES, get_scenario, list_closed_loop
from cstr_sbi.recycle.simulator import nominal_warm_start, deterministic_window

import jax.numpy as jnp
import jax

DATA = pathlib.Path('../data')
FIGURES = pathlib.Path('../figures'); FIGURES.mkdir(exist_ok=True)
SBI_LOGS = pathlib.Path('../sbi-logs'); SBI_LOGS.mkdir(exist_ok=True)
OI = ["#000000","#E69F00","#56B4E9","#009E73","#F0E442","#0072B2","#D55E00","#CC79A7"]
print(f"N_SUMMARIES_SA={N_SUMMARIES_SA}, N_SUMMARIES_SB={N_SUMMARIES_SB}")
print(f"PARAM_NAMES={PARAM_NAMES}")

## 1. Generate S-A Training Data

Same pipeline as nb24 but using `NOMINAL_CTRL_SA` and `compute_summaries(..., "S-A", ...)`,
which produces 72-D summary vectors (10 channels × 6 stats + 12 physics features).

In [ ]:
# Runtime: N_TRAIN=1000 → ~5–15 min; N_TRAIN=15000 → ~75–90 min (set for this study).
N_TRAIN = 15000
print(f"Generating {N_TRAIN} S-A training simulations...")

prior = box_uniform_5d()
rng_np = np.random.default_rng(20260626)
y0_sa = nominal_warm_start("S-A")

thetas_list = []
summaries_list = []
t0 = time.time()

theta_samples = prior.sample((N_TRAIN,)).numpy()

for i, th in enumerate(theta_samples):
    theta_jnp = jnp.array(th, dtype=jnp.float32)
    ts, ys = simulate_trajectory_explicit(
        theta_jnp, NOMINAL_INLET, NOMINAL_CTRL_SA, y0_sa,
        t_final=2.0, n_save=120
    )
    raw = extract_observations_explicit(ys, theta_jnp, NOMINAL_CTRL_SA)
    raw_np = np.asarray(raw)
    t_np = np.asarray(ts)
    if np.isnan(raw_np).any() or np.isinf(raw_np).any():
        continue
    scale = np.maximum(np.max(np.abs(raw_np), axis=0), 1e-6)
    noise = rng_np.normal(0, 0.003 * scale, raw_np.shape)
    s = compute_summaries(raw_np + noise, "S-A", t_np)
    if np.isnan(s).any():
        continue
    thetas_list.append(th)
    summaries_list.append(s)
    if (i + 1) % 500 == 0:
        elapsed = time.time() - t0
        print(f"  {i+1}/{N_TRAIN}  ({elapsed:.0f}s, {len(thetas_list)} valid)")

thetas_sa = np.stack(thetas_list)
summaries_sa = np.stack(summaries_list)
print(f"Valid: {len(thetas_sa)}/{N_TRAIN}")
print(f"thetas shape:    {thetas_sa.shape}")
print(f"summaries shape: {summaries_sa.shape}")
np.savez(DATA / 'wu2003_sbi_train_sa.npz', thetas=thetas_sa, summaries=summaries_sa)
print(f"Saved {DATA / 'wu2003_sbi_train_sa.npz'}")

## 2. Train SNPE on S-A

In [ ]:
from sbi.inference import SNPE
from sbi.utils import posterior_nn

train_data = np.load(DATA / 'wu2003_sbi_train_sa.npz')
thetas_t = torch.tensor(train_data['thetas'], dtype=torch.float32)
summaries_t = torch.tensor(train_data['summaries'], dtype=torch.float32)
print(f"Training data: {thetas_t.shape[0]} samples, {summaries_t.shape[1]}-D summaries (S-A)")

density_estimator = posterior_nn(
    model='nsf',
    hidden_features=128,
    num_transforms=5,
)
inference_sa = SNPE(prior=prior, density_estimator=density_estimator)
inference_sa.append_simulations(thetas_t, summaries_t)

print("Training S-A SNPE (up to 200 epochs, early stopping at 20 without improvement)...")
t0 = time.time()
de_sa = inference_sa.train(
    max_num_epochs=200,
    validation_fraction=0.1,
    stop_after_epochs=20,
    show_train_summary=True,
)
print(f"Done in {time.time()-t0:.0f}s")
posterior_sa = inference_sa.build_posterior(de_sa)
print(f"S-A posterior built: {type(posterior_sa)}")

## 3. Simulation-Based Calibration (SBC) — S-A

In [ ]:
N_SBC = 200   # 500 for publication
N_POST = 100
print(f"Running S-A SBC: {N_SBC} prior samples, {N_POST} posterior samples each...")

sbc_thetas_sa = prior.sample((N_SBC,)).numpy()
sbc_ranks_sa = np.zeros((N_SBC, 5), dtype=int)
rng_sbc = np.random.default_rng(888)
t0 = time.time()

for i, th in enumerate(sbc_thetas_sa):
    theta_jnp = jnp.array(th, dtype=jnp.float32)
    ts, ys = simulate_trajectory_explicit(
        theta_jnp, NOMINAL_INLET, NOMINAL_CTRL_SA, y0_sa,
        t_final=2.0, n_save=120
    )
    raw = extract_observations_explicit(ys, theta_jnp, NOMINAL_CTRL_SA)
    raw_np = np.asarray(raw)
    t_np = np.asarray(ts)
    if np.isnan(raw_np).any():
        sbc_ranks_sa[i] = N_POST // 2
        continue
    scale = np.maximum(np.max(np.abs(raw_np), axis=0), 1e-6)
    noise = rng_sbc.normal(0, 0.003 * scale, raw_np.shape)
    s = compute_summaries(raw_np + noise, "S-A", t_np)
    if np.isnan(s).any():
        sbc_ranks_sa[i] = N_POST // 2
        continue
    x_obs = torch.tensor(s, dtype=torch.float32)
    post_samp = posterior_sa.sample((N_POST,), x=x_obs).numpy()
    for k in range(5):
        sbc_ranks_sa[i, k] = int(np.sum(post_samp[:, k] < th[k]))
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N_SBC}  ({time.time()-t0:.0f}s)")

print(f"SBC done in {time.time()-t0:.0f}s")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3))
n_bins = 10
uniform_count = N_SBC / n_bins
for k, (ax, param) in enumerate(zip(axes, PARAM_NAMES)):
    ax.hist(sbc_ranks_sa[:, k], bins=n_bins, range=(0, N_POST),
            color=OI[k % len(OI)], edgecolor='white', alpha=0.8)
    ax.axhline(uniform_count, ls='--', color='gray', lw=1.5)
    ks_p = scipy_stats.ks_1samp(
        sbc_ranks_sa[:, k] / N_POST, scipy_stats.uniform.cdf
    ).pvalue
    ax.set_title(f"{param}\nKS p={ks_p:.3f}", fontsize=9)
    ax.set_xlabel("Posterior rank")
    if k == 0:
        ax.set_ylabel("Count")
plt.suptitle("SBC Rank Histograms — S-A Posterior", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / 'nb25_sbc_ranks_sa.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved nb25_sbc_ranks_sa.png")

## 4. W12 Headline: S-A vs S-B Posterior Comparison

W12 (`alpha=0.75`, `eta_col=0.80`) is the key compound-fault scenario.  
Under S-B the joint marginal over (alpha, eta_col) is banana-shaped.  
Under S-A the x_D channel breaks the degeneracy, dramatically narrowing the posterior.

In [ ]:
# Load S-B posterior from nb24
sb_pkl_path = SBI_LOGS / 'wu2003_posterior_sb.pkl'
if not sb_pkl_path.exists():
    print(f"WARNING: {sb_pkl_path} not found. Run nb24 first to generate the S-B posterior.")
    posterior_sb = None
else:
    with open(sb_pkl_path, 'rb') as f:
        sb_data = pickle.load(f)
    posterior_sb = sb_data['posterior']
    print(f"Loaded S-B posterior (trained on {sb_data['N_TRAIN']} sims)")

sc_w12 = get_scenario("W12_snowball_compound")
true_th_w12 = np.asarray(sc_w12.theta())
print(f"W12 true theta: alpha={true_th_w12[0]:.2f}, beta_r={true_th_w12[1]:.2f}, "
      f"eta_col={true_th_w12[2]:.2f}, xi_reb={true_th_w12[3]:.2f}, z_A0_eff={true_th_w12[4]:.3f}")

In [ ]:
y0_sb_ws = nominal_warm_start("S-B")
y0_sa_ws = nominal_warm_start("S-A")
rng_c = np.random.default_rng(42)

# S-B observation
t_h_w12_sb, raw_w12_sb = deterministic_window(sc_w12, structure="S-B", y0=y0_sb_ws)
raw_sb_np = np.asarray(raw_w12_sb)
scale_sb = np.maximum(np.max(np.abs(raw_sb_np), axis=0), 1e-6)
noise_sb = rng_c.normal(0, 0.003 * scale_sb, raw_sb_np.shape)
s_sb = compute_summaries(raw_sb_np + noise_sb, "S-B", np.asarray(t_h_w12_sb))

# S-A observation
t_h_w12_sa, raw_w12_sa = deterministic_window(sc_w12, structure="S-A", y0=y0_sa_ws)
raw_sa_np = np.asarray(raw_w12_sa)
scale_sa = np.maximum(np.max(np.abs(raw_sa_np), axis=0), 1e-6)
noise_sa = rng_c.normal(0, 0.003 * scale_sa, raw_sa_np.shape)
s_sa = compute_summaries(raw_sa_np + noise_sa, "S-A", np.asarray(t_h_w12_sa))

# Draw posterior samples
x_sa = torch.tensor(s_sa, dtype=torch.float32)
samp_sa = posterior_sa.sample((2000,), x=x_sa).numpy()

if posterior_sb is not None:
    x_sb = torch.tensor(s_sb, dtype=torch.float32)
    samp_sb = posterior_sb.sample((2000,), x=x_sb).numpy()
else:
    samp_sb = None

print(f"S-A  alpha: mean={samp_sa[:,0].mean():.3f}, 90%CI [{np.percentile(samp_sa[:,0],5):.3f}, {np.percentile(samp_sa[:,0],95):.3f}]")
if samp_sb is not None:
    print(f"S-B  alpha: mean={samp_sb[:,0].mean():.3f}, 90%CI [{np.percentile(samp_sb[:,0],5):.3f}, {np.percentile(samp_sb[:,0],95):.3f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

if samp_sb is not None:
    axes[0].scatter(samp_sb[:, 0], samp_sb[:, 2], alpha=0.05, s=3, color=OI[2], label='S-B posterior')
    axes[0].scatter(true_th_w12[0], true_th_w12[2], color='red', marker='*',
                    s=200, zorder=5, label=f'True (α={true_th_w12[0]:.2f}, η={true_th_w12[2]:.2f})')
    axes[0].set_xlabel("alpha", fontsize=11)
    axes[0].set_ylabel("eta_col", fontsize=11)
    axes[0].set_title("W12: S-B Posterior\n(banana shape — alpha/eta_col degenerate)", fontsize=10)
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, "S-B posterior not available\n(run nb24 first)",
                 ha='center', va='center', transform=axes[0].transAxes)

axes[1].scatter(samp_sa[:, 0], samp_sa[:, 2], alpha=0.05, s=3, color=OI[3], label='S-A posterior')
axes[1].scatter(true_th_w12[0], true_th_w12[2], color='red', marker='*',
                s=200, zorder=5, label=f'True (α={true_th_w12[0]:.2f}, η={true_th_w12[2]:.2f})')
axes[1].set_xlabel("alpha", fontsize=11)
axes[1].set_ylabel("eta_col", fontsize=11)
axes[1].set_title("W12: S-A Posterior\n(narrower — x_D breaks degeneracy)", fontsize=10)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle("W12 Snowball Compound: S-A vs S-B Posterior (alpha vs eta_col)", fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / 'nb25_w12_sa_vs_sb.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved nb25_w12_sa_vs_sb.png")

## 5. Information Value of x_D: 90% CI Width Across All 16 Scenarios

For each scenario we compute the 90% CI width under S-B and S-A posteriors,
then report the relative reduction: `(CI_SB - CI_SA) / CI_SB`.

In [ ]:
n_sc = len(CLOSED_LOOP_NAMES)
print(f"Sampling posteriors for all {n_sc} scenarios under S-A and S-B...")
print("(~10–20 min total)")
ci_widths_sa = np.zeros((n_sc, 5))
ci_widths_sb = np.zeros((n_sc, 5))

for i, sc_name in enumerate(CLOSED_LOOP_NAMES):
    sc = get_scenario(sc_name)
    rng_ci = np.random.default_rng(i + 200)

    # S-A
    t_sa_i, raw_sa_i = deterministic_window(sc, structure="S-A", y0=y0_sa_ws)
    raw_sa_i = np.asarray(raw_sa_i)
    sc_sa = np.maximum(np.max(np.abs(raw_sa_i), axis=0), 1e-6)
    s_sa_i = compute_summaries(
        raw_sa_i + rng_ci.normal(0, 0.003 * sc_sa, raw_sa_i.shape),
        "S-A", np.asarray(t_sa_i)
    )
    samp_sa_i = posterior_sa.sample(
        (500,), x=torch.tensor(s_sa_i, dtype=torch.float32)
    ).numpy()
    ci_widths_sa[i] = (
        np.percentile(samp_sa_i, 95, axis=0) - np.percentile(samp_sa_i, 5, axis=0)
    )

    # S-B (only if posterior is available)
    if posterior_sb is not None:
        t_sb_i, raw_sb_i = deterministic_window(sc, structure="S-B", y0=y0_sb_ws)
        raw_sb_i = np.asarray(raw_sb_i)
        sc_sb = np.maximum(np.max(np.abs(raw_sb_i), axis=0), 1e-6)
        s_sb_i = compute_summaries(
            raw_sb_i + rng_ci.normal(0, 0.003 * sc_sb, raw_sb_i.shape),
            "S-B", np.asarray(t_sb_i)
        )
        samp_sb_i = posterior_sb.sample(
            (500,), x=torch.tensor(s_sb_i, dtype=torch.float32)
        ).numpy()
        ci_widths_sb[i] = (
            np.percentile(samp_sb_i, 95, axis=0) - np.percentile(samp_sb_i, 5, axis=0)
        )
    else:
        ci_widths_sb[i] = np.nan

    print(f"  {sc_name}: alpha CI  S-A={ci_widths_sa[i,0]:.3f}  S-B={ci_widths_sb[i,0]:.3f}")

print("Done.")

In [ ]:
# Summary table: relative reduction in CI width
reduction = (ci_widths_sb - ci_widths_sa) / (ci_widths_sb + 1e-8)

df_ci = pd.DataFrame(index=CLOSED_LOOP_NAMES)
for k, p in enumerate(PARAM_NAMES):
    df_ci[f"SB_{p}"]  = ci_widths_sb[:, k]
    df_ci[f"SA_{p}"]  = ci_widths_sa[:, k]
    df_ci[f"red_{p}"] = reduction[:, k]

print("90% CI width and relative reduction (alpha, eta_col):")
print(
    df_ci[['SB_alpha', 'SA_alpha', 'red_alpha',
           'SB_eta_col', 'SA_eta_col', 'red_eta_col']].to_string(float_format="{:.3f}".format)
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_pos = np.arange(n_sc)
w = 0.35

for ax, k, param in [
    (axes[0], 0, 'alpha'),
    (axes[1], 2, 'eta_col'),
]:
    ax.bar(x_pos - w / 2, ci_widths_sb[:, k], w,
           label='S-B', color=OI[2], alpha=0.8, edgecolor='white')
    ax.bar(x_pos + w / 2, ci_widths_sa[:, k], w,
           label='S-A', color=OI[3], alpha=0.8, edgecolor='white')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([n[:8] for n in CLOSED_LOOP_NAMES],
                        rotation=45, ha='right', fontsize=8)
    ax.set_ylabel("90% CI width", fontsize=10)
    ax.set_title(f"{param}: S-A vs S-B posterior uncertainty", fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)

plt.suptitle("Information Value of x_D: 90% CI Width Comparison", fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES / 'nb25_ci_width_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved nb25_ci_width_comparison.png")

## 6. Save S-A Posterior

In [ ]:
save_path_sa = SBI_LOGS / 'wu2003_posterior_sa.pkl'
with open(save_path_sa, 'wb') as f:
    pickle.dump({
        'posterior': posterior_sa,
        'inference': inference_sa,
        'density_estimator': de_sa,
        'N_TRAIN': len(thetas_sa),
    }, f)
print(f"S-A posterior saved to {save_path_sa}")
print()
print("Key findings:")
print(f"  W12 alpha 90%CI:  S-B={ci_widths_sb[CLOSED_LOOP_NAMES.index('W12_snowball_compound'),0]:.3f}")
print(f"                    S-A={ci_widths_sa[CLOSED_LOOP_NAMES.index('W12_snowball_compound'),0]:.3f}")
print(f"  W12 eta_col 90%CI: S-B={ci_widths_sb[CLOSED_LOOP_NAMES.index('W12_snowball_compound'),2]:.3f}")
print(f"                     S-A={ci_widths_sa[CLOSED_LOOP_NAMES.index('W12_snowball_compound'),2]:.3f}")
print()
print("=== Notebook 25 complete ===")